In [ ]:
# Install dependencies
!pip uninstall -y -q torch torchvision torchaudio cudf easyocr numba-cuda dask-cuda pylibraft-cu12 raft-dask-cu12 pylibcudf-cu12 rmm-cu12 quantecon cuvs-cu12 ucxx-cu12 cuml-cu12 cuda-python cuml libcuml libcuvs libraft numba ydata-profiling google-colab
!pip install -q torch==2.5.1+cu121 torchvision==0.20.1+cu121 --index-url https://download.pytorch.org/whl/cu121
!pip install -q torch>=2.1.0 transformers>=5.15.0 datasets>=2.14.0 numpy>=1.24.0 matplotlib>=3.7.0 scipy>=1.10.0 papermill>=2.7.0 testbook>=0.4.2
!pip install -q -U transformers


In [ ]:
import os
import sys
import time
import traceback
import torch
# Fix allocator fragmentation (exp16r OOM: reserved-but-unallocated
# grew to 1.1GB across alphas) — set BEFORE any CUDA use.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# KAGGLEHUB_CACHE: persist models across sessions
os.environ["KAGGLEHUB_CACHE"] = "/kaggle/working/.model_cache"
os.makedirs("/kaggle/working/.model_cache", exist_ok=True)
WORKING = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.expanduser("~/.kaggle_test")
sys.path.insert(0, WORKING)
num_gpus = torch.cuda.device_count()
if num_gpus >= 2:
    device_L, device_S = "cuda:0", "cuda:1"
    dtype_L = dtype_S = torch.float32
elif num_gpus == 1:
    device_L = device_S = "cuda:0"
    dtype_L = torch.float16
    dtype_S = torch.float32
else:
    device_L = device_S = "cpu"
    dtype_L = dtype_S = torch.float32
if num_gpus > 0:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"torch {torch.__version__} | L={device_L}, S={device_S}")
RES = os.path.join(WORKING, "results")
os.makedirs(RES, exist_ok=True)
# Resolve models: kernel_sources (prev commit outputs) -> working -> HF
def resolve_model(hf_id):
    model_dir = "qwen35-08b" if "0.8B" in hf_id else "qwen35-4b"
    # 1. Check each slug in /kaggle/input/ — listdir only (isdir lies on mounts)
    input_dir = "/kaggle/input"
    try:
        slugs = os.listdir(input_dir)
        print(f"resolve {hf_id}: slugs={slugs}")
        for slug in slugs:
            p_sub = os.path.join(input_dir, slug)
            try:
                contents = os.listdir(p_sub)
            except OSError:
                continue
            if model_dir in contents:
                p = os.path.join(p_sub, model_dir)
                print(f"Found {hf_id} at {p} (via {slug})")
                return p
    except Exception as e:
        print(f"DEBUG: /kaggle/input/ check failed: {e}")
    # 2. Check WORKING
    wf = os.path.join(WORKING, model_dir)
    if os.path.isdir(wf):
        print(f"Found {hf_id} at {wf}")
        return wf
    # 3. Return HF ID - will download + persist
    print(f"Need to download {hf_id} from HF")
    return hf_id
# Load + save for persistence (calls from_pretrained, then save_pretrained if from HF)
def load_model(cls, hf_id, **kw):
    path = resolve_model(hf_id)
    model = cls.from_pretrained(path, **kw)
    # Persist if loaded from HF
    if path == hf_id:
        dst = os.path.join(WORKING, "qwen35-08b" if "0.8B" in hf_id else "qwen35-4b")
        if not os.path.isdir(dst):
            print(f"Saving {hf_id} to {dst}...")
            model.save_pretrained(dst)
            ensure_tokenizer(hf_id, dst)
            print(f"Saved")
    return model
# Ensure tokenizer files exist in a persisted model dir
def ensure_tokenizer(hf_id, dst):
    import transformers
    if not os.path.exists(os.path.join(dst, "tokenizer.json")):
        print(f"Saving tokenizer for {hf_id}...")
        transformers.AutoTokenizer.from_pretrained(hf_id).save_pretrained(dst)
SMALL = resolve_model("Qwen/Qwen3.5-0.8B")
LARGE = resolve_model("Qwen/Qwen3.5-4B")


In [ ]:
import torch
from transformers import DynamicCache
from transformers.cache_utils import (  # type: ignore[import-not-found]
    DynamicLayer,
    LinearAttentionLayer,
)
def compute_jacobian_for_tokens(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    token_set: list[int],
) -> torch.Tensor:
    """Compute Jacobian of logits w.r.t. hidden state at layer_idx.
    Args:
        model: Qwen3_5ForCausalLM instance.
        layer_idx: Layer index (0-based) to compute Jacobian at.
        input_ids: [1, S] token IDs.
        attention_mask: [1, S] attention mask.
        token_set: list of token IDs to compute gradients for.
    Returns:
        J: [|U|, D] Jacobian matrix (unnormalized).
    """
    model.eval()
    S = input_ids.shape[1]
    prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
    h = _extract_hidden_state(model, input_ids, attention_mask, layer_idx)
    # Match model dtype to avoid mismatch in decoder (fp16 models on GPU)
    model_dtype = next(model.parameters()).dtype
    h = h.to(model_dtype).detach().requires_grad_(True)
    # Re-enable grad: the decoder forward must build a graph over the leaf
    # h even when the caller runs under no_grad (e.g. eval loops in exp18/19).
    with torch.enable_grad():
        logits = _run_decoder_pass(model, prefix_cache, h, S, layer_idx)
    J = _compute_jacobian_rows(logits, h, token_set)
    return J
def _run_prefix_pass(
    model, input_ids: torch.Tensor, attention_mask: torch.Tensor
) -> DynamicCache:
    """Prefix pass: all tokens except last -> cache."""
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids[:, :-1],
            attention_mask=attention_mask[:, :-1],
            use_cache=True,
            output_hidden_states=False,
        )
    return outputs.past_key_values
def _extract_hidden_state(model, input_ids, attention_mask, layer_idx) -> torch.Tensor:
    """Full pass: extract hidden state at layer_idx for last token."""
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            use_cache=False,
        )
    # hidden_states[0] = embeddings, [l+1] = output of layer l
    h = outputs.hidden_states[layer_idx + 1][:, -1:, :]
    return h  # [1, 1, D]
def _deep_clone_cache(cache):
    """Deep clone cache, ensuring all tensors are independent copies.
    ponytail: copy.deepcopy works but is slow. Manual clone for speed + safety.
    """
    new_cache = DynamicCache()
    for idx in range(len(cache.layers)):
        layer = cache.layers[idx]
        if isinstance(layer, DynamicLayer):
            new_k = [k.clone() if k is not None else None for k in layer.keys]
            new_v = [v.clone() if v is not None else None for v in layer.values]
            new_cache.layers.append(DynamicLayer(keys=new_k, values=new_v))
        else:
            new_conv = {k: v.clone() for k, v in layer.conv_states.items()}
            new_recur = {k: v.clone() for k, v in layer.recurrent_states.items()}
            new_cache.layers.append(
                LinearAttentionLayer(conv_states=new_conv, recurrent_states=new_recur)
            )
    return new_cache
def _run_decoder_pass(
    model, prefix_cache, h: torch.Tensor, S: int, layer_idx: int
) -> torch.Tensor:
    """Decode h through remaining layers using prefix cache."""
    # Clone cache tensors independently to avoid in-place mutation
    cache_clone = _deep_clone_cache(prefix_cache)
    text_model = model.model  # Qwen3_5TextModel
    pos = torch.full((4, 1, 1), S - 1, dtype=torch.long, device=h.device)
    text_position_ids = pos[0]
    rope_position_ids = pos[1:]
    position_embeddings = text_model.rotary_emb(h, rope_position_ids)
    x = h
    n_layers = model.config.num_hidden_layers
    start = layer_idx + 1
    causal_mask_mapping = _build_causal_masks(
        text_model, text_position_ids, cache_clone, x
    )
    for i in range(start, n_layers):
        layer_type = model.config.layer_types[i]
        attn_mask = causal_mask_mapping[layer_type]
        x = text_model.layers[i](
            x,
            position_embeddings=position_embeddings,
            attention_mask=attn_mask,
            position_ids=text_position_ids,
            past_key_values=cache_clone,
        )
    x = text_model.norm(x)
    logits = model.lm_head(x[:, -1, :])
    return logits
def _build_causal_masks(text_model, text_pos, cache, x):
    """Build causal mask mapping for a single token with cache."""
    from transformers.masking_utils import (
        create_causal_mask,
        create_recurrent_attention_mask,
    )
    dummy_embeds = x.new_zeros(1, 1, text_model.config.hidden_size)
    mask_kwargs = {
        "config": text_model.config,
        "inputs_embeds": dummy_embeds,
        "attention_mask": None,
        "past_key_values": cache,
        "position_ids": text_pos,
    }
    return {
        "full_attention": create_causal_mask(**mask_kwargs),
        "linear_attention": create_recurrent_attention_mask(**mask_kwargs),
    }
def _compute_jacobian_rows(
    logits: torch.Tensor, h: torch.Tensor, token_set: list[int]
) -> torch.Tensor:
    """Compute Jacobian rows via loop over token_set."""
    logits_subset = logits[0, token_set]  # [|U|]
    D = h.shape[-1]
    J = torch.zeros(len(token_set), D, device=h.device, dtype=h.dtype)
    for i in range(len(token_set)):
        grad = torch.autograd.grad(
            outputs=logits_subset[i],
            inputs=h,
            retain_graph=(i < len(token_set) - 1),
        )[0]
        J[i] = grad[0, 0, :]
    return J
def softmax_jacobian(J: torch.Tensor, p: torch.Tensor) -> torch.Tensor:
    """Gauge-invariant Jacobian: J^(p) = diag(p) - pp^T @ J.
    J^(p)_i = p_i * (J_i - J_bar_p),  J_bar_p = sum_k p_k * J_k
    Removes the common-mode shift (gauge) from raw Jacobian.
    """
    J_bar = (p[:, None] * J).sum(dim=0, keepdim=True)  # [1, D]
    return p[:, None] * (J - J_bar)  # [K, D]
def uniform_centered(J: torch.Tensor) -> torch.Tensor:
    """Uniform centering: J - mean(J). Removes average shift."""
    return J - J.mean(dim=0, keepdim=True)
def _last_layer_jacobian(h: torch.Tensor, norm: torch.nn.Module) -> torch.Tensor:
    """Analytical J = d h_norm / d h at the final norm (no backward passes).
    Both Qwen3_5RMSNorm and nn.LayerNorm are handled. J is evaluated at the
    norm INPUT (h), never at h_norm — evaluating the formula at the norm
    output gives a matrix with ~0.87 relative error (no longer LayerNorm-like
    statistics). Qwen3.5's final norm is Qwen3_5RMSNorm:
      h_norm = (h * rsqrt(mean(h^2) + eps) * (1 + w))
      J = r * (diag(1+w) - xhat x^T / D),  r = rsqrt(mean(h^2)+eps), xhat = r*h
    LayerNorm: J = (I - 1/D - xhat xhat^T / D) / std, rows scaled by w.
    """
    x = h.reshape(1, -1).float()  # [1, D], the norm INPUT (row vector)
    D = x.shape[-1]
    eps = float(getattr(norm, "eps", None) or 1e-5)
    w = getattr(norm, "weight", None)
    dev = x.device
    if getattr(norm, "bias", None) is not None:  # LayerNorm family
        mu = x.mean(dim=-1, keepdim=True)
        std = (x.var(dim=-1, unbiased=False, keepdim=True) + eps).sqrt()
        xhat = (x - mu) / std  # [1, D]
        J = (
            torch.eye(D, device=dev)
            - torch.ones(D, D, device=dev) / D
            - xhat.T @ xhat / D
        ) / std.squeeze(0)
        if w is not None:
            J = J * w.float().squeeze(0).unsqueeze(1)  # J = diag(w) @ J
        return J
    # RMSNorm family (Qwen3_5RMSNorm, Llama-style).
    r = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + eps)
    xhat = x * r  # [D]
    # y_j = (1+w_j) r x_j; J = r * diag(1+w) @ (I - xhat xhat^T / D)
    # (derived: J_ij = g_j (r d_ij - r^3 x_i x_j / D) = g_j r (d_ij - xhat_i xhat_j / D))
    g = 1.0 + w.float().squeeze(0) if w is not None else None
    M = torch.eye(D, device=dev) - xhat.T @ xhat / D
    if g is None:
        return r.squeeze(0) * M
    return r.squeeze(0) * torch.diag(g) @ M
def compute_hidden_jacobian(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    create_graph: bool = False,
    verbose: bool = False,
) -> torch.Tensor:
    """Compute Jacobian of normalized last hidden state w.r.t. hidden state at layer_idx.
    J_norm = d(h_norm) / d(h_{layer_idx}), shape [D, D].
    Independent of lm_head — computed once, reused across W updates.
    Then J_S = lm_head.weight[token_set] @ J_norm is differentiable through W.
    For last_layer: J_norm is LayerNorm Jacobian (analytical, no backward passes).
    Args:
        create_graph: if True, J_norm has grad_fn.
                      h is kept connected to model params (no detach). Full forward pass.
                      Costs ~D backward passes per sample.
        verbose: print progress during J_norm computation.
    """
    model.eval()
    n_layers = model.config.num_hidden_layers
    if create_graph:
        # Keep gradient graph alive: h is connected to model params (+ any hooks)
        S = input_ids.shape[1]
        # Build prefix cache FIRST — before any forward involving hooks.
        prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
        # Full forward to get h at layer_idx — connected to all model params + hooks
        with torch.enable_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True,
                use_cache=False,
            )
        h = outputs.hidden_states[layer_idx + 1][:, -1:, :]  # [1, 1, D], connected!
        model_dtype = next(model.parameters()).dtype
        h = h.to(model_dtype)  # preserve grad graph
        if h.grad_fn is None:
            # Fully frozen graph: no trainable path reaches this layer
            # (e.g. linear_attention pair with no LoRA patch, or a
            # no_grad caller). Make h a leaf so the decoder pass below
            # still yields a differentiable h_norm; J_norm values are
            # identical to the connected case.
            h = h.detach().requires_grad_(True)
        if layer_idx == n_layers - 1:
            with torch.enable_grad():
                h_norm = model.model.norm(h[:, -1, :])  # [1, D]
        else:
            with torch.enable_grad():
                h_norm = _run_decoder_to_norm(
                    model.model, prefix_cache, h, S, layer_idx
                )
        # [1, 1, D] — _compute_jacobian_rows_full indexes grad[0, 0, :].
        # enable_grad: autograd.grad needs grad mode ON even when the
        # caller is inside torch.no_grad().
        with torch.enable_grad():
            J_norm = _compute_jacobian_rows_full(
                h_norm, h, create_graph=True, verbose=verbose
            )
        return J_norm
    # Last layer: analytical norm Jacobian, no backward passes.
    if layer_idx == n_layers - 1:
        with torch.no_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True,
                use_cache=False,
            )
            h = outputs.hidden_states[layer_idx + 1][:, -1:, :]  # [1, 1, D]
        return _last_layer_jacobian(h, model.model.norm)
    # General case: decode through remaining layers + norm
    S = input_ids.shape[1]
    prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
    h = _extract_hidden_state(model, input_ids, attention_mask, layer_idx)
    model_dtype = next(model.parameters()).dtype
    h = h.to(model_dtype).detach().requires_grad_(True)
    # Re-enable grad: h_norm must carry a grad_fn over h even if the caller
    # is inside torch.no_grad() (e.g. eval loops in exp16r/exp17). The row
    # loop below (autograd.grad) ALSO needs grad mode ON — a no_grad caller
    # otherwise gets "element 0 ... does not have a grad_fn".
    with torch.enable_grad():
        h_norm = _run_decoder_to_norm(model.model, prefix_cache, h, S, layer_idx)
        J_norm = _compute_jacobian_rows_full(
            h_norm, h, create_graph=create_graph, verbose=verbose
        )
    return J_norm
def _run_decoder_to_norm(
    text_model, prefix_cache, h: torch.Tensor, S: int, layer_idx: int
) -> torch.Tensor:
    """Decode h through remaining layers, return norm(x) without lm_head."""
    cache_clone = _deep_clone_cache(prefix_cache)
    pos = torch.full((4, 1, 1), S - 1, dtype=torch.long, device=h.device)
    text_position_ids = pos[0]
    rope_position_ids = pos[1:]
    position_embeddings = text_model.rotary_emb(h, rope_position_ids)
    x = h
    n_layers = text_model.config.num_hidden_layers
    start = layer_idx + 1
    causal_mask_mapping = _build_causal_masks(
        text_model, text_position_ids, cache_clone, x
    )
    for i in range(start, n_layers):
        layer_type = text_model.config.layer_types[i]
        attn_mask = causal_mask_mapping[layer_type]
        x = text_model.layers[i](
            x,
            position_embeddings=position_embeddings,
            attention_mask=attn_mask,
            position_ids=text_position_ids,
            past_key_values=cache_clone,
        )
    return text_model.norm(x[:, -1, :])
def _compute_jacobian_rows_full(
    output: torch.Tensor,
    h: torch.Tensor,
    create_graph: bool = False,
    verbose: bool = False,
) -> torch.Tensor:
    """Full [D, D] Jacobian via autograd.grad."""
    import time as _time
    D = h.shape[-1]
    out_flat = output.flatten()
    grads = []
    start = _time.time()
    for i in range(D):
        grad = torch.autograd.grad(
            outputs=out_flat[i],
            inputs=h,
            retain_graph=(i < D - 1),
            create_graph=create_graph,
        )[0]
        grads.append(grad[0, 0, :])
        if verbose and i % 128 == 0:
            elapsed = _time.time() - start
            rate = i / elapsed if elapsed > 0 else 999
            eta = elapsed * (D / i) - elapsed if i > 0 else 0.0
            print(f"        J_norm: {i}/{D} ({rate:.0f}/s, eta={eta:.0f}s)")
    J = torch.stack(grads)
    elapsed_total = _time.time() - start
    if verbose:
        print(f"        J_norm done: {D}/{D} in {elapsed_total:.1f}s")
    return J


In [ ]:
import hashlib
import os
import torch
_gramian_cache: dict[str, torch.Tensor] = {}
_GRAMIAN_DIR = os.path.join("results", ".gramians")
def jacobian_bridge(
    J_L: torch.Tensor,
    J_S: torch.Tensor,
    E_L: torch.Tensor,
    E_S: torch.Tensor,
    probs_L: torch.Tensor,
    threshold: float = 1e-6,
) -> float:
    """Compute Jacobian bridge loss between large and small models.
    The bridge chain: J_L @ E_L^T @ E_S @ (E_S^T @ E_S)^{-1}.
    The inverse Gramian (E_S^T @ E_S)^{-1} corrects for non-orthogonal
    embeddings, so identity (J_L == J_S, E_L == E_S) yields loss ~0.
    """
    G_inv = _inverse_gramian(E_S)
    J_L_proj = _project_to_vocab(J_L, E_L)
    J_S_target = _project_to_small(J_L_proj, E_S, G_inv)
    mask = _valid_mask(J_S, J_S_target, threshold)
    if mask.sum() == 0:
        return 1.0
    return _weighted_cosine_loss(J_S, J_S_target, probs_L, mask)
def _project_to_vocab(J_L: torch.Tensor, E_L: torch.Tensor) -> torch.Tensor:
    """Project J_L [|U|, D_L] through E_L^T -> [|U|, V]."""
    return J_L.float() @ E_L.float().T
def _project_to_small(
    J_L_proj: torch.Tensor, E_S: torch.Tensor, G_inv: torch.Tensor
) -> torch.Tensor:
    """Project [|U|, V] through E_S @ G_inv -> [|U|, D_S].
    G_inv = (E_S^T @ E_S)^{-1} corrects non-orthogonal embeddings.
    """
    return J_L_proj @ E_S.float() @ G_inv
def _inverse_gramian(E: torch.Tensor) -> torch.Tensor:
    """Compute (E^T @ E)^{-1}, cached on disk per embedding hash."""
    key = _gramian_key(E)
    if key not in _gramian_cache:
        path = os.path.join(_GRAMIAN_DIR, f"{key}.pt")
        if os.path.exists(path):
            _gramian_cache[key] = torch.load(path, weights_only=True)
        else:
            _gramian_cache[key] = _compute_inverse_gramian(E)
            os.makedirs(_GRAMIAN_DIR, exist_ok=True)
            torch.save(_gramian_cache[key], path)
    return _gramian_cache[key].to(E.device)
def _gramian_key(E: torch.Tensor) -> str:
    """Stable hash of an embedding matrix for disk caching.
    Uses first/last rows as representative content — hashing the full
    248k×1024 matrix would be prohibitively expensive.
    """
    header = f"{tuple(E.shape)}|{str(E.device)}".encode()
    data = header + E[0].cpu().numpy().tobytes() + E[-1].cpu().numpy().tobytes()
    h = hashlib.sha256(data).hexdigest()[:16]
    return f"g_{h}"
def _compute_inverse_gramian(E: torch.Tensor) -> torch.Tensor:
    """Compute (E^T @ E)^{-1} via pseudoinverse for numerical stability."""
    G = E.float().T @ E.float()
    return torch.linalg.pinv(G)
def _valid_mask(
    J_S: torch.Tensor, J_target: torch.Tensor, threshold: float
) -> torch.Tensor:
    """Mask rows where either norm < threshold."""
    n1 = J_S.float().norm(dim=1)
    n2 = J_target.float().norm(dim=1)
    return (n1 > threshold) & (n2 > threshold)
def _weighted_cosine_loss(
    J_S: torch.Tensor,
    J_target: torch.Tensor,
    probs: torch.Tensor,
    mask: torch.Tensor,
) -> float:
    """Weighted cosine dissimilarity loss."""
    cos_sim = torch.nn.functional.cosine_similarity(
        J_S.float()[mask], J_target[mask], dim=1
    )
    w = probs[mask]
    w = w / (w.sum() + 1e-8)
    return (w * (1.0 - cos_sim)).sum().item()


In [ ]:
import torch
def compute_esg(
    jacobians: list[torch.Tensor],
    n_components: int = 32,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Compute ESG from a list of Jacobian matrices.
    Args:
        jacobians: list of N Jacobian matrices, each [|U|, D].
        n_components: number of top eigenvalues/vectors to return.
    Returns:
        eigenvalues: [n_components] sorted descending.
        eigenvectors: [n_components, |U|] corresponding eigenvectors.
    """
    G = _build_gramian(jacobians)
    eigenvalues, eigenvectors = _spectral_decomposition(G, n_components)
    return eigenvalues, eigenvectors
def _build_gramian(jacobians: list[torch.Tensor]) -> torch.Tensor:
    """G = (1/N) * sum_i (J_i @ J_i^T)."""
    U = jacobians[0].shape[0]
    device = jacobians[0].device
    G = torch.zeros(U, U, device=device, dtype=torch.float)
    for J in jacobians:
        G += J.float() @ J.float().T
    G /= len(jacobians)
    return G
def _spectral_decomposition(
    G: torch.Tensor, n_components: int
) -> tuple[torch.Tensor, torch.Tensor]:
    """Eigen decomposition, sorted descending."""
    eigenvalues, eigenvectors = torch.linalg.eigh(G)
    # Sort descending
    idx = eigenvalues.argsort(descending=True)
    eigenvalues = eigenvalues[idx][:n_components]
    eigenvectors = eigenvectors[:, idx][:, :n_components].T
    return eigenvalues, eigenvectors


In [ ]:
from collections import Counter
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
def uniform_layer_mapping(n_layers_large: int, n_layers_small: int) -> dict[int, int]:
    """Uniform layer mapping from large to small model layers."""
    mapping = {}
    for i in range(n_layers_large):
        j = round(i * (n_layers_small - 1) / (n_layers_large - 1))
        mapping[i] = j
    return mapping
def get_layer_types(model: AutoModelForCausalLM) -> list[str]:
    """Get layer types from model config (full_attention / linear_attention)."""
    return list(model.config.layer_types)  # type: ignore[attr-defined]
_DOMAIN_DATASETS = {
    "wiki": ("allenai/c4", "en", "train", "text"),
    "math": ("openai/gsm8k", "main", "train", "question"),
    "code": ("neyralabs/apex-agents", None, "train", "prompt"),
    "chat": ("HuggingFaceH4/ultrachat_200k", None, "train_sft", "messages"),
}
def _load_dataset_for_domain(
    domain: str,
) -> tuple[str, str | None, str, str]:
    """Return (dataset_id, config, split, text_field) for a domain.
    Falls back to wiki (C4) if the target dataset is unavailable.
    """
    ds_id, cfg, split, field = _DOMAIN_DATASETS[domain]
    try:
        kw: dict[str, object] = {"split": split, "streaming": True}
        if cfg is not None:
            kw["name"] = cfg
        load_dataset(ds_id, **kw)  # type: ignore[call-overload]
        return ds_id, cfg, split, field
    except Exception:
        print(f"  Warning: domain {domain} ({ds_id}) unavailable, falling back to C4")
        return _DOMAIN_DATASETS["wiki"]
def _extract_text_from_sample(
    sample: dict,
    field: str,
) -> str:
    """Extract text from a dataset sample, handling different formats."""
    if field == "messages":
        # Chat format: list of {role, content}
        texts = [m.get("content", "") for m in sample.get("messages", [])]
        return " ".join(texts)
    val = sample.get(field, "")
    if isinstance(val, list):
        # e.g. code_contests solutions: list of strings
        return "\n".join(str(v) for v in val)
    if isinstance(val, dict):
        return str(val)
    return val
def load_calibration_data(
    model_name: str,
    max_length: int = 512,
    min_length: int = 256,
    n_samples: int = 4,
    domain: str = "wiki",
) -> list[dict[str, torch.Tensor]]:
    """Load and tokenize calibration data.
    domain: 'wiki' (C4), 'math' (GSM8K), 'code' (apex-agents), 'chat' (ultrachat).
    Falls back to wiki if the target dataset is unavailable.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    ds_id, cfg, split, field = _load_dataset_for_domain(domain)
    kw: dict[str, object] = {"split": split, "streaming": True}
    if cfg is not None:
        kw["name"] = cfg
    dataset = load_dataset(ds_id, **kw)  # type: ignore[call-overload]
    samples = []
    for raw in dataset:
        text = _extract_text_from_sample(raw, field)
        if not text:
            continue
        tokens = tokenizer(
            text, max_length=max_length, truncation=True, return_tensors="pt"
        )
        if tokens["input_ids"].shape[1] >= min_length:
            samples.append(tokens)
        if len(samples) >= n_samples:
            break
    return samples
def build_global_token_set(
    model: AutoModelForCausalLM,
    calibration_samples: list[dict[str, torch.Tensor]],
    K: int = 64,
) -> list[int]:
    """Build global token set from top-K logits across all samples."""
    token_counter: Counter[int] = Counter()
    model.eval()  # type: ignore[attr-defined]
    device = next(model.parameters()).device  # type: ignore[attr-defined]
    with torch.no_grad():
        for sample in calibration_samples:
            inputs = {
                "input_ids": sample["input_ids"].to(device),
                "attention_mask": sample["attention_mask"].to(device),
            }
            outputs = model(**inputs)  # type: ignore[operator]
            logits = outputs.logits[:, -1, :]
            top_idx = logits.topk(K, dim=-1).indices[0]
            token_counter.update(top_idx.tolist())
    U = [tid for tid, _ in token_counter.most_common(K)]
    return sorted(U)
def load_models(
    large_name: str, small_name: str, dtype: torch.dtype = torch.float16
) -> tuple[torch.nn.Module, torch.nn.Module]:
    """Load large and small models with eager attention."""
    kwargs = {
        "torch_dtype": dtype,
        "attn_implementation": "eager",
    }
    large = AutoModelForCausalLM.from_pretrained(large_name, **kwargs)
    large.eval()  # type: ignore[attr-defined]
    small = AutoModelForCausalLM.from_pretrained(small_name, **kwargs)
    small.eval()  # type: ignore[attr-defined]
    return large, small
def extract_embeddings(
    model: AutoModelForCausalLM,
) -> torch.Tensor:
    """Extract embedding matrix from model."""
    return model.model.embed_tokens.weight.detach()  # type: ignore[attr-defined]


In [ ]:
import os
# Max depth under input_root to search (producer puts cache at depth 2:
# results/targets; 4 covers any reasonable layout).
_MAX_DEPTH = 4
def _has_pt(d: str) -> bool:
    try:
        return any(f.endswith(".pt") for f in os.listdir(d))
    except OSError:
        return False
def _walk(root: str, depth: int):
    """Yield dirs under root (listdir-based), depth-limited."""
    try:
        entries = sorted(os.listdir(root))
    except OSError:
        return
    for e in entries:
        p = os.path.join(root, e)
        if _has_pt(p):
            yield p
        if depth > 1:
            yield from _walk(p, depth - 1)
def find_targets_dir(preferred: str, input_root: str = "/kaggle/input") -> str:
    """Return the dir holding the target-cache *.pt files.
    Order: `preferred` (local path / runner-injected), then a listdir walk
    of every mount under `input_root`. Raises FileNotFoundError with the
    mount inventory when nothing is found (no silent fallback, no listdir crash).
    """
    if _has_pt(preferred):
        return preferred
    for hit in _walk(input_root, _MAX_DEPTH):
        return hit
    mounts = []
    try:
        for d in sorted(os.listdir(input_root)):
            try:
                mounts.append(f"{d}/:{os.listdir(os.path.join(input_root, d))[:4]}")
            except OSError:
                mounts.append(f"{d}/:<unreadable>")
    except OSError:
        mounts.append("<no input mount>")
    raise FileNotFoundError(
        f"No target cache .pt in {preferred} nor under {input_root}/ "
        f"(mounts: {mounts}). Run Stage 0 first — it must publish results/targets/*.pt."
    )


In [ ]:
import json
import math
import os
import re
import time
from collections.abc import Callable
import torch
from transformers import AutoModelForCausalLM  # type: ignore[import-not-found]
def _layer_order(pairs: list[str], device: str) -> dict[str, list[str]]:
    late_to_early = sorted(pairs, key=lambda p: int(p.split("_")[1]), reverse=True)
    if device == "cpu":
        wanted = [23, 15, 7]
        subset = [p for p in late_to_early if int(p.split("_")[1]) in wanted]
        if not subset:
            subset = late_to_early[:7]
        return {"late_to_early": subset}
    return {
        "late_to_early": late_to_early,
        "early_to_late": list(reversed(late_to_early)),
    }
def main_17loradistill(
    small_model: str = "Qwen/Qwen3.5-0.8B",
    device: str = "auto",
    max_length: int = 64,
    min_length: int = 32,
    targets_dir: str = "results/targets",
    results_dir: str = "results",
):
    torch.manual_seed(42)
    start = time.time()
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cpu":
        torch.set_num_threads(4)
    print(f"Device: {device}")
    os.makedirs(results_dir, exist_ok=True)
    # -- Calibration data -------------------------------------------------
    n_samples = 2 if device == "cpu" else 4
    print(f"Loading {n_samples} calibration samples...")
    samples = load_calibration_data(
        small_model,
        max_length=max_length,
        min_length=min_length,
        n_samples=n_samples,
    )
    # -- Load target cache ------------------------------------------------
    # FUSE-safe, layout-independent (prism.cache_io): preferred dir, else
    # listdir walk of /kaggle/input — no hardcoded slug/subpath.
    targets_dir = find_targets_dir(targets_dir)
    print(f"Using target cache at {targets_dir}")
    # -- Load target cache ------------------------------------------------
    cache_files = sorted(
        f
        for f in os.listdir(targets_dir)
        if f.endswith(".pt") and not f.startswith(".")
    )
    if not cache_files:
        raise FileNotFoundError(
            f"No target cache .pt files found in {targets_dir}. Run Stage 0 first."
        )
    cache_by_pair: dict[str, list[dict]] = {}
    for cf in cache_files:
        data = torch.load(os.path.join(targets_dir, cf), weights_only=True)
        pair = data["pair"]
        # Tag with the sample index stored in the filename ({pair}_{si}.pt)
        m = re.search(r"_(\d+)\.pt$", cf)
        data["_si"] = int(m.group(1)) if m else -1
        cache_by_pair.setdefault(pair, []).append(data)
    # Sort each pair's targets by sample index (filename sort breaks at si>=10)
    for pair in cache_by_pair:
        cache_by_pair[pair].sort(key=lambda t: t["_si"])
    all_pairs = sorted(cache_by_pair.keys(), key=lambda p: int(p.split("_")[1]))
    orders = _layer_order(all_pairs, device)
    steps = 100 if device.startswith("cuda") else 40
    r = 4
    lr = 1e-3
    kl_weight = 0.1
    j_budget = 900.0 if device.startswith("cuda") else 300.0
    per_order_results = {}
    for order_name, pair_seq in orders.items():
        print(f"\n{'=' * 60}")
        print(f"Order: {order_name} ({len(pair_seq)} pairs)")
        print(f"{'=' * 60}")
        # Load fresh model for this order
        mS = AutoModelForCausalLM.from_pretrained(
            small_model,
            torch_dtype=torch.float32,
            attn_implementation="eager",
            low_cpu_mem_usage=True,
        ).to(device)  # type: ignore[arg-type]
        mS.eval()
        # Cache original logits (updated after each layer)
        orig_logits_cache: list[torch.Tensor] = []
        with torch.no_grad():
            for s in samples[:n_samples]:
                ids = s["input_ids"].to(device)
                attn = s["attention_mask"].to(device)
                orig_logits_cache.append(
                    mS(input_ids=ids, attention_mask=attn).logits[:, -1, :]
                )
        per_layer_results: list[dict] = []
        for pair_key in pair_seq:
            targets = cache_by_pair[pair_key]
            l_S = int(pair_key.split("_")[1])
            token_set = targets[0]["token_set"]
            n_use = min(n_samples, len(targets))
            print(f"\n  Pair {pair_key} (l_S={l_S}, {n_use} targets)")
            layer_start = time.time()
            # Freeze all model params
            for p in mS.parameters():
                p.requires_grad = False
            # LoRA on lm_head: W_eff = W + B_lh @ A_lh, B: [V,r], A: [r,D]
            V, D = mS.lm_head.weight.shape
            A_lh = torch.nn.Parameter(torch.randn(r, D, device=device) * 0.02)
            B_lh = torch.nn.Parameter(torch.zeros(r, V, device=device).T)  # [V, r]
            # LoRA on target layer's attention o_proj (full-attention layers
            # only; GDN linear-attention layers have out_proj, no o_proj).
            # Qwen3.5's o_proj is NON-square [out, in] = [1024, 2048], so the
            # LoRA delta is applied to o_proj's INPUT x [*, in] by patching
            # the Linear's forward: delta = B @ (A @ x). Dimensionally exact
            # for any in/out and immune to positional-vs-kwargs callers.
            layer_mod = mS.model.layers[l_S]
            o_proj_mod = None
            if hasattr(layer_mod, "self_attn"):
                o_proj_mod = getattr(layer_mod.self_attn, "o_proj", None)
            A_attn: torch.nn.Parameter | None = None
            B_attn: torch.nn.Parameter | None = None
            hook_remove: Callable[..., None] | None = None
            if o_proj_mod is not None:
                out_dim, in_dim = o_proj_mod.weight.shape
                A_attn = torch.nn.Parameter(
                    torch.randn(r, in_dim, device=device) * 0.02
                )
                B_attn = torch.nn.Parameter(torch.zeros(out_dim, r, device=device))
                # This is how A_attn/B_attn connect to the grad graph: with
                # create_graph=True the J's h is built through the patched
                # o_proj, so dL/dJ_norm contracted with dJ_norm/dh reaches
                # o_proj.
                _orig_fwd = o_proj_mod.forward
                def _patched_o_proj(
                    x: torch.Tensor,
                    _b: torch.Tensor = B_attn,
                    _a: torch.Tensor = A_attn,
                    _f: Callable = _orig_fwd,
                ) -> torch.Tensor:
                    delta = (_b @ _a @ x.transpose(-1, -2)).transpose(-1, -2)
                    return _f(x) + delta
                o_proj_mod.forward = _patched_o_proj
                def _restore(
                    _m: torch.nn.Module = o_proj_mod, _f: Callable = _orig_fwd
                ) -> None:
                    _m.forward = _f
                hook_remove = _restore
            trainable = [A_lh, B_lh]
            if A_attn is not None:
                trainable += [A_attn, B_attn]  # type: ignore[list-item]
            for p in trainable:
                p.requires_grad = True
            optimizer = torch.optim.Adam(trainable, lr=lr)
            # -- J_norm refresh plan (amortized) ---------------------------
            # Probe: first create_graph J. Measures t_J (for k) and probes
            # OOM. On OOM the layer degrades: detached J for lm_head bridge,
            # o_proj trained by KL only.
            t_J = None
            use_fresh = False
            k = steps  # no refresh while probing
            J_norms: dict[int, torch.Tensor] = {}
            first_s = samples[0]
            ids_0 = first_s["input_ids"].to(device)
            attn_0 = first_s["attention_mask"].to(device)
            try:
                t0 = time.time()
                J_norms[0] = compute_hidden_jacobian(
                    mS, l_S, ids_0, attn_0, create_graph=True, verbose=True
                )
                t_J = time.time() - t0
                use_fresh = True
                if A_attn is not None and t_J is not None:
                    # One refresh ~3*t_J: J build + 2nd-order backward (HVP)
                    # through the retained D-row graph.
                    n_affordable = int(j_budget // (n_use * 3 * max(t_J, 1e-3)))
                    k = max(1, min(10, math.ceil(steps / max(n_affordable, 1))))
            except torch.cuda.OutOfMemoryError:
                print(
                    f"    [warn] create_graph J OOM on {pair_key} — "
                    "o_proj falls back to KL-only, lm_head keeps detached-J bridge"
                )
                torch.cuda.empty_cache()
                J_norms = {}
            print(f"    J refresh: k={k}, use_fresh={use_fresh}, t_J={t_J:.1f}s")
            # smoke_j (first pair only, ~1 extra 2nd-order backward): prove the
            # create_graph graph reaches o_proj. Without this, a silent graph
            # break (e.g. model.eval() killing grads through frozen layers)
            # would train o_proj by KL only forever with no warning.
            if order_name == next(iter(orders)) and A_attn is not None and J_norms:
                try:
                    g_tgt = targets[0]["J_tgt_p"].to(device).float()
                    # Frozen W is fine for a liveness probe: the o_proj path
                    # runs through J_norms[0]'s h-graph, not the lm_head delta.
                    probe_loss = torch.nn.functional.cosine_similarity(
                        (mS.lm_head.weight[token_set] @ J_norms[0]).float(),
                        g_tgt,
                        dim=1,
                    ).mean()
                    probe_loss.backward()
                    gA = (
                        A_attn.grad.abs().sum().item()
                        if A_attn.grad is not None
                        else 0.0
                    )
                    gB = (
                        B_attn.grad.abs().sum().item()
                        if B_attn is not None and B_attn.grad is not None
                        else 0.0
                    )
                    ok = gA > 0 or gB > 0
                    print(
                        f"    [smoke_j] dL/dA_attn={gA:.2e}, dL/dB_attn={gB:.2e} "
                        f"-> {'OK' if ok else 'graph dead'}"
                    )
                    if not ok:
                        print(
                            "    [warn] create_graph graph does not reach o_proj "
                            "— o_proj stays KL-only for this layer"
                        )
                except Exception as e:  # noqa: BLE001 - probe must not kill the run
                    print(f"    [warn] smoke_j failed: {e}")
                # The probe's backward() consumed J_norms[0]'s create_graph
                # graph; a step-0 backward on it would be a second backward
                # (RuntimeError). Replace it with a detached J — the real
                # create_graph refresh happens at step % k == 0, step > 0.
                if 0 in J_norms:
                    del J_norms[0]
                    try:
                        J_norms[0] = compute_hidden_jacobian(mS, l_S, ids_0, attn_0)
                    except torch.cuda.OutOfMemoryError:
                        torch.cuda.empty_cache()
                        print(
                            "    [warn] detached J fill OOM after smoke_j — "
                            "si=0 bridge off until first refresh"
                        )
            for step in range(steps):
                step_start = time.time()
                optimizer.zero_grad()
                total_bridge = 0.0
                total_kl = 0.0
                n_bridge = 0
                # Effective lm_head weight: W + B_lh @ A_lh
                W_eff = mS.lm_head.weight + B_lh @ A_lh  # [V, D]
                for si in range(n_use):
                    s = samples[si]
                    tgt = targets[si]
                    # KL leash — full forward (includes o_proj LoRA via hook)
                    ids_s = s["input_ids"].to(device)
                    attn_s = s["attention_mask"].to(device)
                    out = mS.model(input_ids=ids_s, attention_mask=attn_s)
                    hidden = out.last_hidden_state[:, -1, :]  # [1, D]
                    lm_b = mS.lm_head.bias if mS.lm_head.bias is not None else 0
                    logits_ed = hidden @ W_eff.T + lm_b  # [1, V]
                    p_ed = torch.softmax(logits_ed.float(), dim=-1)
                    with torch.no_grad():
                        p_orig = torch.softmax(orig_logits_cache[si].float(), dim=-1)
                    kl_loss = (
                        p_ed * torch.log(p_ed.clamp_min(1e-12) / (p_orig + 1e-12))
                    ).mean()
                    loss = kl_weight * kl_loss
                    total_kl += kl_loss.item()
                    # Bridge — only for matched targets
                    if tgt.get("_si") == si:
                        J_tgt_p = tgt["J_tgt_p"].to(device)
                        p_L = tgt["p"].to(device)
                        if si not in J_norms:
                            # one-time fill: detached J (lm_head bridge only)
                            J_norms[si] = compute_hidden_jacobian(
                                mS, l_S, ids_s, attn_s
                            )
                        elif use_fresh and step > 0 and step % k == 0:
                            # refresh: fresh J with grad graph — bridge backward
                            # also delivers o_proj gradient (HVP).
                            del J_norms[si]
                            try:
                                J_norms[si] = compute_hidden_jacobian(
                                    mS, l_S, ids_s, attn_s, create_graph=True
                                )
                            except (
                                torch.cuda.OutOfMemoryError,
                                RuntimeError,
                            ) as e:
                                # n fresh graphs on a refresh step can OOM
                                # even after a successful 1-graph probe ->
                                # degrade the layer (stale-J bridge for
                                # lm_head, o_proj KL-only), never kill the
                                # kernel (docs #16).
                                if "out of memory" in str(e).lower():
                                    torch.cuda.empty_cache()
                                print(
                                    f"    [warn] refresh J OOM on {pair_key} "
                                    f"(si={si}): layer degrades to stale-J + "
                                    "o_proj KL-only"
                                )
                                use_fresh = False
                                J_norms = {}
                                break
                        if J_norms.get(si) is not None:
                            # J_S is (re)built from whatever J is current:
                            # detached on fill/stale steps, create_graph on
                            # refresh steps (-> o_proj bridge gradient).
                            J_S = W_eff[token_set] @ J_norms[si]
                            J_S_p = softmax_jacobian(J_S, p_L)
                            cos_sim = torch.nn.functional.cosine_similarity(
                                J_S_p.float(), J_tgt_p.float(), dim=1
                            )
                            bridge_loss = (1.0 - cos_sim).mean()
                            loss = loss + bridge_loss
                            total_bridge += bridge_loss.item()
                            n_bridge += 1
                    loss.backward()
                optimizer.step()
                step_time = time.time() - step_start
                eta = (steps - step - 1) * step_time
                if (step + 1) % 5 == 0 or step == 0:
                    print(
                        f"    step {step + 1}/{steps}: "
                        f"bridge={total_bridge / max(n_bridge, 1):.4f}, "
                        f"kl={total_kl:.4f}, step={step_time:.0f}s, eta={eta:.0f}s"
                    )
            # -- Freeze lora deltas into weights ---------------------------
            delta_lm_final = B_lh @ A_lh  # [V, D]
            with torch.no_grad():
                mS.lm_head.weight.data += delta_lm_final
            if o_proj_mod is not None and A_attn is not None and B_attn is not None:
                # B[out, r] @ A[r, in] = [out, in] == o_proj.weight shape.
                o_proj_mod.weight.data += B_attn @ A_attn
            # Cleanup BEFORE eval: the delta is now frozen in the weights;
            # leaving the live patch on would double it in the eval forward.
            J_norms.clear()
            del A_lh, B_lh
            if A_attn is not None:
                del A_attn, B_attn
            if hook_remove is not None:
                hook_remove()
            # -- Evaluate bridge (no grad, create_graph=False) -------------
            bridge_vals = []
            with torch.no_grad():
                W_final = mS.lm_head.weight  # frozen
                for si in range(n_use):
                    s = samples[si]
                    tgt = targets[si]
                    if tgt.get("_si") != si:
                        continue
                    ids = s["input_ids"].to(device)
                    attn_s = s["attention_mask"].to(device)
                    J_tgt_p = tgt["J_tgt_p"].to(device)
                    p_L = tgt["p"].to(device)
                    J_norm_eval = compute_hidden_jacobian(mS, l_S, ids, attn_s)
                    J_S_ed = W_final[token_set] @ J_norm_eval
                    J_S_ed_p = softmax_jacobian(J_S_ed, p_L)
                    cos = (
                        torch.nn.functional.cosine_similarity(
                            J_S_ed_p.float(), J_tgt_p.float(), dim=1
                        )
                        .mean()
                        .item()
                    )
                    bridge_vals.append(cos)
            # KL to original
            kl_sum = 0.0
            with torch.no_grad():
                for si in range(min(n_samples, len(orig_logits_cache))):
                    s = samples[si]
                    ids = s["input_ids"].to(device)
                    attn_s = s["attention_mask"].to(device)
                    logits_ed = mS(input_ids=ids, attention_mask=attn_s).logits[
                        :, -1, :
                    ]
                    p_ed = torch.softmax(logits_ed.float(), dim=-1)
                    p_orig = torch.softmax(orig_logits_cache[si].float(), dim=-1)
                    kl = (
                        (p_ed * torch.log(p_ed.clamp_min(1e-12) / (p_orig + 1e-12)))
                        .mean()
                        .item()
                    )
                    kl_sum += kl
            kl_avg = kl_sum / min(n_samples, len(orig_logits_cache))
            layer_result = {
                "pair": pair_key,
                "bridge_mean": sum(bridge_vals) / len(bridge_vals)
                if bridge_vals
                else 0,
                "KL_ed_orig": kl_avg,
                "j_refresh_k": k,
                "j_fresh": use_fresh,
                "t_J_first_s": t_J,
            }
            layer_time = time.time() - layer_start
            print(f"    Layer {pair_key} done in {layer_time:.0f}s")
            print(f"    bridge={layer_result['bridge_mean']:.4f}, KL={kl_avg:.4f}")
            per_layer_results.append(layer_result)
            # Update orig logits for next layer (cumulative edits)
            with torch.no_grad():
                for si, s in enumerate(samples[:n_samples]):
                    ids = s["input_ids"].to(device)
                    attn_s = s["attention_mask"].to(device)
                    orig_logits_cache[si] = mS(
                        input_ids=ids, attention_mask=attn_s
                    ).logits[:, -1, :]
        per_order_results[order_name] = {
            "pairs": pair_seq,
            "per_layer": per_layer_results,
        }
    # -- Save --------------------------------------------------------------
    result = {
        "description": (
            "Per-layer LoRA distillation guided by Jacobian bridge. For each layer pair "
            "(large→small), attaches LoRA r=4 to lm_head and attention o_proj of l_S. "
            "J_norm = d(h_norm)/d(h_lS) is per-sample: first create_graph J probes OOM and "
            "measures t_J, then refreshes every k-th step (k = amortized from budget "
            "900s-GPU/300s-CPU, capped at 10). Fresh J is computed with create_graph=True "
            "so the bridge loss also differentiates through h (HVP, no D^3 tensor) and "
            "reaches the o_proj LoRA; between refreshes lm_head gets bridge gradient from "
            "the stale J and o_proj is trained by the KL leash only. OOM -> o_proj KL-only. "
            "After each pair both lora deltas are frozen into weights. Target sample index "
            "is parsed from the stage0 cache filename; mismatch -> bridge skipped for that "
            "sample (KL unaffected). Two orders: late-to-early and early-to-late."
        ),
        "per_order": per_order_results,
        "r": r,
        "steps": steps,
        "kl_weight": kl_weight,
        "j_budget_s": j_budget,
        "device": device,
        "elapsed_seconds": time.time() - start,
    }
    path = os.path.join(results_dir, "exp17_lora_distill.json")
    result["params"] = {
        "small_model": small_model,
        "device": device,
        "max_length": max_length,
        "min_length": min_length,
        "targets_dir": targets_dir,
    }
    try:
        with open(path, "w") as f:
            json.dump(result, f, indent=2)
    except OSError as e:
        print(f"Failed to write {path}: {e}")
        raise
    print(f"\nSaved to {path}")
    print(f"Total time: {time.time() - start:.0f}s")


In [ ]:
# ---------- run exp17_lora_distill ----------
import gc
# Hard cleanup from previous kernel version (same session, different code)
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
import inspect
sig = inspect.signature(main_17loradistill)
kwargs = {}
params = list(sig.parameters)
if 'small_model' in params: kwargs['small_model'] = SMALL
if 'large_model' in params: kwargs['large_model'] = LARGE
if 'model' in params: kwargs['model'] = SMALL
if 'device_L' in params: kwargs['device_L'] = device_L
if 'device_S' in params: kwargs['device_S'] = device_S
if 'device' in params: kwargs['device'] = device_S
if 'results_dir' in params: kwargs['results_dir'] = RES
if 'targets_dir' in params: kwargs['targets_dir'] = os.path.join(RES, 'targets')
# Remove conflicting keys: if both model and small_model, prefer specific
if 'small_model' in kwargs and 'model' in kwargs: del kwargs['model']
# Memory diagnostic before experiment
if torch.cuda.is_available():
    print(f'cuda:0 alloc={torch.cuda.memory_allocated(0)/1024**3:.1f}GiB reserved={torch.cuda.memory_reserved(0)/1024**3:.1f}GiB')
    print(f'cuda:1 alloc={torch.cuda.memory_allocated(1)/1024**3:.1f}GiB reserved={torch.cuda.memory_reserved(1)/1024**3:.1f}GiB')
t0 = time.time()
try:
    main_17loradistill(**kwargs)
    elapsed = time.time() - t0
    print(f"[OK] exp17_lora_distill {elapsed:.0f}s")
    # Persist models (in case experiment loaded from HF but didn't save)
    try:
        dst_s = os.path.join(WORKING, "qwen35-08b")
        dst_l = os.path.join(WORKING, "qwen35-4b")
        if os.path.isdir(dst_s):
            print(f"Small model persisted at {dst_s}")
        if os.path.isdir(dst_l):
            print(f"Large model persisted at {dst_l}")
    except Exception:
        pass
except Exception as e:
    print(f"[FAIL] exp17_lora_distill: {e}")
    traceback.print_exc()
print("\nResults:")
if os.path.isdir(RES):
    try:
        for fn in sorted(os.listdir(RES)):
            print(f"  {fn} {os.path.getsize(os.path.join(RES, fn)):,} bytes")
    except OSError:
        pass
